# AutoGeneS deconvolution and downstream analysis

In [ ]:
from pathlib import Path
import itertools

import autogenes as ag
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'DejaVu Sans'
mpl.rcParams['axes.unicode_minus'] = False

## 1. Mean expression reference by cell subtype

In [ ]:
adata_anno = sc.read_h5ad('../adata_anno_cell_subtype_re.h5ad')
adata_qc = sc.read_h5ad('../adata_qc.h5ad')

adata_proc = sc.pp.normalize_total(adata_qc, target_sum=1e4, copy=True)
adata_proc = adata_proc[adata_anno.obs_names].copy()
adata_proc.obs['cell_subtype'] = adata_proc.obs_names.map(
    adata_anno.obs['cell_subtype'].to_dict()
)

In [ ]:
X = adata_proc.X.toarray() if hasattr(adata_proc.X, 'toarray') else adata_proc.X
expr_df = pd.DataFrame(X, index=adata_proc.obs_names, columns=adata_proc.var_names)
mean_expr = expr_df.groupby(adata_proc.obs['cell_subtype']).mean().T
mean_expr.to_csv('adata_proc_mean_by_subtype.csv')

## 2. AutoGeneS optimization and NuSVR deconvolution

In [ ]:
expr = pd.read_csv('adata_proc_mean_by_subtype.csv', index_col=0)
data_bulk_raw = pd.read_csv('gene_symbol_fpkm_star_kirc.csv', index_col=0)

In [ ]:
ag.init(expr.T)
ag.optimize(
    ngen=10000,
    seed=0,
    nfeatures=2400,
    mode='fixed',
    offspring_size=100,
    verbose=True,
)
ag.select(index=0)

In [ ]:
def normalize_proportions(data, copy):
    data_copy = data.copy() if copy else data
    data_copy[data_copy < 0] = 0
    for row in data_copy.index:
        row_sum = data_copy.loc[row].sum()
        data_copy.loc[row] = np.divide(data_copy.loc[row], row_sum)
    return data_copy


coef_nusvr = ag.deconvolve(data_bulk_raw.T, model='nusvr')
proportions_NuSVR = normalize_proportions(
    pd.DataFrame(
        data=coef_nusvr,
        columns=expr.columns,
        index=data_bulk_raw.columns,
    ),
    copy=False,
)
proportions_NuSVR.to_csv('proportions_NuSVR_10000.csv')

## 3. NuSVR subtype-proportion heatmap

In [ ]:
sns.set_theme(style='whitegrid')
df = pd.read_csv('proportions_NuSVR_10000.csv', index_col=0).astype(float)
g = sns.clustermap(
    df,
    cmap='viridis',
    standard_scale=1,
    figsize=(12, 10),
    xticklabels=True,
    yticklabels=False,
    linewidths=0,
    cbar_kws={'label': 'Z-score of Proportion'},
)
g.fig.suptitle('Cell Type Proportions Heatmap (NuSVR)', y=1.02)
g.fig.savefig('NuSVR_10000_col.pdf', bbox_inches='tight')
plt.close(g.fig)

## 4. Non-epithelial subtype Spearman correlations

In [ ]:
base = Path('.')
out_dir = base / 'NuSVR_10000_spearman_correlation'
out_dir.mkdir(exist_ok=True)

df = pd.read_csv(base / 'proportions_NuSVR_10000.csv', index_col=0).astype(float)
non_epi_cols = [column for column in df.columns if not column.startswith('Epi_')]
df_non_epi = df.loc[:, non_epi_cols]

std = df_non_epi.std(axis=0, ddof=0)
constant_cols = std.index[std <= 0].tolist()
df_non_epi = df_non_epi.drop(columns=constant_cols) if constant_cols else df_non_epi

corr = df_non_epi.corr(method='spearman')
corr.to_csv(out_dir / 'proportions_NuSVR_10000_raw_spearman_rho_matrix_non_epi.csv')

In [ ]:
rows = []
for subtype_a, subtype_b in itertools.combinations(df_non_epi.columns, 2):
    result = spearmanr(df_non_epi[subtype_a], df_non_epi[subtype_b], nan_policy='omit')
    rows.append({
        'subtype_a': subtype_a,
        'subtype_b': subtype_b,
        'rho': float(result.statistic),
        'p': float(result.pvalue),
        'n_samples': int(df_non_epi[[subtype_a, subtype_b]].dropna().shape[0]),
    })

pair_df = pd.DataFrame(rows)
pair_df['q_bh'] = multipletests(pair_df['p'].values, method='fdr_bh')[1]
pair_df = pair_df.sort_values(['q_bh', 'p', 'rho'], ascending=[True, True, False])
pair_df.to_csv(
    out_dir / 'proportions_NuSVR_10000_raw_spearman_pairwise_long_non_epi.csv',
    index=False,
)
pair_df.sort_values('rho', ascending=False).head(50).to_csv(
    out_dir / 'top50_positive_non_epi_spearman_pairs.csv',
    index=False,
)
pair_df.sort_values('rho').head(50).to_csv(
    out_dir / 'top50_negative_non_epi_spearman_pairs.csv',
    index=False,
)

In [ ]:
sns.set_theme(style='white')
g = sns.clustermap(
    corr,
    cmap='vlag',
    center=0,
    vmin=-1,
    vmax=1,
    figsize=(12, 12),
    xticklabels=True,
    yticklabels=True,
    linewidths=0,
    cbar_kws={'label': 'Spearman rho'},
)
g.fig.suptitle('AutoGeneS NuSVR non-epithelial subtype abundance correlation', y=1.02)
g.fig.savefig(
    out_dir / 'proportions_NuSVR_10000_raw_spearman_rho_clustermap_non_epi.pdf',
    bbox_inches='tight',
)
g.fig.savefig(
    out_dir / 'proportions_NuSVR_10000_raw_spearman_rho_clustermap_non_epi.svg',
    bbox_inches='tight',
)
plt.close(g.fig)

In [ ]:
summary = pd.DataFrame([
    {'item': 'input_file', 'value': 'proportions_NuSVR_10000.csv'},
    {'item': 'n_samples', 'value': str(df_non_epi.shape[0])},
    {'item': 'n_non_epi_subtypes_used', 'value': str(df_non_epi.shape[1])},
    {
        'item': 'constant_non_epi_columns_dropped',
        'value': ';'.join(constant_cols) if constant_cols else 'none',
    },
    {'item': 'correlation_method', 'value': 'Spearman correlation on non-epithelial NuSVR proportions'},
    {'item': 'multiple_testing', 'value': 'Benjamini-Hochberg FDR across non-epithelial subtype pairs'},
])
summary.to_csv(
    out_dir / 'proportions_NuSVR_10000_spearman_summary_non_epi.csv',
    index=False,
)